# 11 — Silver Suppliers & Bidders

**Objetivo:** construir proveedores canónicos y una relación proceso–proponente sin forzar todavía la identidad bidder→supplier.

Este notebook conserva los identificadores necesarios para `13_entity_resolution.ipynb`, evita la expansión observada en Bronze y excluye del Silver analítico general campos de contacto/nombres sensibles que no son necesarios para la resolución determinística.

## Invariantes

- `suppliers.codigo` fue validado en M1 como PK.
- `bidders.id_procedimiento -> processes.id_del_proceso` tiene alta cobertura, pero el lado Processes requería canonicalización.
- El join directo `bidders.codigo_proveedor -> suppliers.codigo` tuvo 0% de cobertura en M1: **no se fuerza aquí**.
- `process_versions.parquet` de notebook 10 permite construir un crosswalk `procedure_id -> process_id` sin multiplicar filas.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files
from argos.config.silver import load_silver_spec
from argos.silver.suppliers_bidders import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

DATASETS, _ = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

check10 = pd.read_parquet(paths.reports / "silver" / "10_validation.parquet")
blocking10 = check10.loc[check10["blocking"]]
assert not blocking10.empty and blocking10["passed"].all(), "Notebook 10 no está cerrado."

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)
{
    "suppliers": FILES["suppliers"].name,
    "bidders": FILES["bidders"].name,
    "implementation": BUILD_IMPLEMENTATION,
    "schema_version": SILVER_SPEC.schema_version,
    "duckdb_threads": SILVER_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": SILVER_SPEC.runtime.duckdb_memory_limit,
}

## Construcción

Salidas:

- `suppliers.parquet` — `1 fila = 1 supplier_code`.
- `bidder_participations.parquet` — preserva la población Bronze de proponentes.
- `procedure_process_map.parquet` — crosswalk sin expansión desde `procedure_id` a `process_id`.
- `bidder_process_links.parquet` — relación canónica proceso–proponente, con población ambigua/unmatched explícita.

In [ ]:
from argos.silver.suppliers_bidders import build_suppliers_bidders_silver

result = build_suppliers_bidders_silver(
    FILES["suppliers"],
    FILES["bidders"],
    suppliers_spec=DATASETS["suppliers"],
    bidders_spec=DATASETS["bidders"],
    column_profile_path=paths.reports / "profiling" / "column_profile.parquet",
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    silver_spec=SILVER_SPEC,
    date_order=PROJECT_CONFIG["temporal"]["date_order"],
    sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)
{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}

## Routing de privacidad

La exclusión es arquitectónica: los campos explícitamente sensibles no entran al Silver analítico general. Los identificadores `supplier_code`, `supplier_nit`, `bidder_code` y `bidder_nit` se conservan porque son necesarios para lineage/entity resolution, no porque sean features autorizadas.

In [ ]:
result.routing.groupby(["dataset", "route"]).size().rename("columns").reset_index()

In [ ]:
result.routing.loc[result.routing["route"] == "excluded_sensitive", [
    "dataset", "raw_column", "canonical_column", "semantic_type", "reason"
]].sort_values(["dataset", "canonical_column"])

## Validación bloqueante

In [ ]:
result.validation

In [ ]:
from argos.common.duckdb import connect
from argos.common.sql import sql_string

con = connect(
    temp_directory=paths.root / ".cache" / "duckdb",
    threads=SILVER_SPEC.runtime.duckdb_threads,
    memory_limit=SILVER_SPEC.runtime.duckdb_memory_limit,
)
try:
    link_status = con.execute(f"""
        SELECT process_link_status, COUNT(*) AS links, SUM(source_row_count) AS source_rows
        FROM read_parquet({sql_string(result.artifacts['bidder_process_links'])})
        GROUP BY process_link_status ORDER BY links DESC
    """).fetchdf()
finally:
    con.close()
link_status

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("11 ✅ PASS" if result.ready_for_next_notebook else "11 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_next_notebook, "Notebook 11 remains OPEN; resolve blocking checks before 12."

**Siguiente:** `12_silver_events.ipynb`. La identidad bidder→supplier sigue pendiente intencionalmente hasta `13`.